# Sediment and morphology

**What this shows:** the options of the `Sediment` component: transport formulae and
calibration, morphological acceleration, avalanching, bed composition, hard layers,
groundwater and prescribed bed changes.

**Prerequisites:** [Tutorial 5: Choosing model settings](../tutorial/05_model_settings.ipynb).

**You will learn:**

- how to switch on sediment transport and bed updating, and calibrate them
- how `morfac` and the morphology time window work
- how to describe uniform or graded sediment
- how to add non-erodible layers, groundwater flow and prescribed bed updates

**Data used:** small input files created in the notebook.

## Setup

In [1]:
import shutil
from pathlib import Path

import numpy as np
from pydantic import ValidationError
from rompy.logging import config as logging_config
from rompy_xbeach.components.sediment import Sediment
from rompy_xbeach.types import XBeachDataBlob

logging_config.update(level="WARNING")  # rompy logs every step, show warnings only

OUT_DIR = Path("_output") / "sediment_and_morphology"
shutil.rmtree(OUT_DIR, ignore_errors=True)
OUT_DIR.mkdir(parents=True)


def show(component):
    """Print the XBeach parameters a component writes to params.txt."""
    for key, value in component.get(destdir=OUT_DIR).items():
        print(f"{key} = {value}")

## 1. Transport and morphology switches

Both are off in XBeach by default. `True` switches them on with default settings.

In [2]:
show(Sediment(sedtrans=True, morphology=True))

sedtrans = 1
morphology = 1


## 2. Transport formulation and calibration

`SedimentTransport` chooses the transport formula and the wave-related calibration
factors. `facua` sets onshore transport by wave skewness and asymmetry, which is a
common calibration parameter for dune and beach profiles.

In [3]:
from rompy_xbeach.components.sediment.transport import SedimentTransport

show(
    Sediment(
        sedtrans=SedimentTransport(
            form="vanthiel_vanrijn",
            facua=0.15,
            bdslpeffmag="roelvink_total",
        )
    )
)

sedtrans = 1
bdslpeffmag = roelvink_total
facua = 0.15
form = vanthiel_vanrijn


## 3. Morphological acceleration and the time window

`morfac` multiplies bed changes, so one hour of hydrodynamics gives `morfac` hours of
bed change. `morstart` delays morphology until the hydrodynamics have spun up, and
`morstop` ends it. Both are given in morphological time.

In [4]:
from rompy_xbeach.components.sediment.morphology import Morphology

show(Sediment(morphology=Morphology(morfac=10.0, morstart=3600.0, morstop=36000.0)))

morphology = 1
morfac = 10.0
morstart = 3600.0
morstop = 36000.0


## 4. Avalanching

Slopes steeper than a critical value collapse. Different thresholds apply above
(`dryslp`) and below (`wetslp`) the water line, with `hswitch` as the dividing depth.

In [5]:
show(Sediment(morphology=Morphology(dryslp=1.0, wetslp=0.3, hswitch=0.1)))

morphology = 1
dryslp = 1.0
hswitch = 0.1
wetslp = 0.3


## 5. Bed composition

A single sediment class is described by its grain sizes. Graded sediment uses one
value per class and `ngd` classes. The lists must have matching lengths, and
D90 must exceed D50.

In [6]:
from rompy_xbeach.components.sediment.composition import BedComposition

show(Sediment(bed_composition=BedComposition(D50=0.0003, D90=0.0005, por=0.4)))

D50 = 0.0003
D90 = 0.0005
por = 0.4


In [7]:
show(
    Sediment(
        bed_composition=BedComposition(
            ngd=2, D50=[0.0002, 0.0008], D90=[0.0003, 0.0012], nd=5, dzg1=0.05
        )
    )
)

ngd = 2
nd = 5
D50 = 0.0002 0.0008
D90 = 0.0003 0.0012
dzg1 = 0.05


In [8]:
try:
    BedComposition(D50=0.0005, D90=0.0003)
except ValidationError as error:
    print(error.errors()[0]["msg"])

Value error, D90[0] (0.0003) must be greater than D50[0] (0.0005)


## 6. Non-erodible layers

Rock, seawalls or reefs are handled with a file giving the erodible sediment thickness
at each grid point. Where it reaches zero the bed cannot erode further.

In [9]:
thickness = np.full((110, 115), 5.0)
thickness[:, 90:] = 0.0  # a hard revetment near the back of the domain
ne_layer = OUT_DIR / "source_ne_layer.txt"
np.savetxt(ne_layer, thickness, fmt="%.1f")

show(
    Sediment(
        morphology=Morphology(struct=True, ne_layer=XBeachDataBlob(source=ne_layer))
    )
)

morphology = 1
struct = 1
ne_layer = source_ne_layer.txt


## 7. Groundwater flow

Groundwater affects swash infiltration and gravel beach profiles. Switch it on in the
physics (`Physics(gwflow=True)`) and describe the aquifer here.

In [10]:
from rompy_xbeach.components.sediment.groundwater import GroundwaterFlow

show(Sediment(groundwater=GroundwaterFlow(aquiferbot=-5.0, gw0=0.0, kx=0.001)))

aquiferbot = -5.0
gw0 = 0.0
kx = 0.001


## 8. Prescribed bed updates

`BedUpdate` controls bed layering during morphology, and can impose bed levels at
given times from a file, for example to represent nourishments or dredging.

In [11]:
from rompy_xbeach.components.sediment.bed import BedUpdate

show(Sediment(bed_update=BedUpdate(frac_dz=0.7, split=1.01, merge=0.01)))

frac_dz = 0.7
merge = 0.01
split = 1.01


## 9. A complete setup

A typical storm-erosion configuration for a sandy beach:

In [12]:
show(
    Sediment(
        sedtrans=SedimentTransport(form="vanthiel_vanrijn", facua=0.1),
        morphology=Morphology(morfac=5.0, morstart=3600.0, wetslp=0.3),
        bed_composition=BedComposition(D50=0.0003, D90=0.0005),
    )
)

sedtrans = 1
facua = 0.1
form = vanthiel_vanrijn
morphology = 1
morfac = 5.0
morstart = 3600.0
wetslp = 0.3
D50 = 0.0003
D90 = 0.0005


## Summary

| Sub-component | Controls |
|---|---|
| `sedtrans` (`SedimentTransport`) | Transport formula, wave effects, calibration |
| `morphology` (`Morphology`) | Bed updating, `morfac`, avalanching, hard layers |
| `bed_composition` (`BedComposition`) | Grain sizes, classes and bed layers |
| `numerics`, `q3d` | Transport numerics and quasi-3D transport |
| `groundwater` (`GroundwaterFlow`) | Aquifer and infiltration, with `Physics(gwflow=True)` |
| `bed_update` (`BedUpdate`) | Layer splitting and prescribed bed levels |